### Importing Libraries

In [54]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import pandas as pd
import pickle
from config import ROOT_DIR
from src.models.baseline_model import create_baseline_recommender,create_recency_baseline
from src.models.als_model import train_als_model, create_als_recommender
from src.evaluation.map_metric import evaluate_all

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [55]:
cleaned_df_folder = "input/cleaned"
def load_from_parquet(cleaned_df_name):
    print("Loading data from Parquet...")
    return pd.read_parquet(cleaned_df_folder + "/" + cleaned_df_name + ".parquet")

In [56]:
cols_needed = ['t_dat', 'customer_id', 'article_id']

train_transactions = pd.read_parquet(
    "input/cleaned/train_transactions.parquet", 
    columns=cols_needed
)

In [57]:
#Loading training datasets
# train_transactions = load_from_parquet("train_transactions")
train_articles = load_from_parquet("train_articles")
train_customers = load_from_parquet("train_customers")

#Loading validation dataset
val_transactions = load_from_parquet("val_transactions")


Loading data from Parquet...
Loading data from Parquet...
Loading data from Parquet...


In [58]:

print("Columns in train_articles:")
print(train_articles.columns.tolist())
print()
print("Columns in train_customers:")
print(train_customers.columns.tolist())
print()
print("Columns in train_transactions:")
print(train_transactions.columns.tolist())
print()

Columns in train_articles:
['article_id', 'product_code', 'product_type_no', 'product_group_name', 'graphical_appearance_no', 'colour_group_code', 'perceived_colour_value_id', 'perceived_colour_master_id', 'department_no', 'index_code', 'index_group_no', 'section_no', 'garment_group_no']

Columns in train_customers:
['customer_id', 'FN', 'club_member_status', 'fashion_news_frequency', 'age']

Columns in train_transactions:
['t_dat', 'customer_id', 'article_id']



In [59]:
# Initialize the baseline recommender to get the top-k recommendations.
my_baseline_model = create_baseline_recommender(train_transactions)

# Evaluate the baseline model using MAP@K metric on the validation transactions.
evaluate_all(my_baseline_model, val_transactions)

Generating predictions for 68,984 validation customers...
Preparing evaluation matrices...
Computing metrics...


{'MAP@12': 0.0028985916189694194,
 'HitRate@12': 0.021120839614983185,
 'NDCG@12': 0.0053114661238347575}

In [61]:
# Evaluate the recency baseline model using MAP@K metric on the validation transactions.
recency_model = create_recency_baseline(train_transactions)
evaluate_all(recency_model, val_transactions)


Generating predictions for 68,984 validation customers...
Preparing evaluation matrices...
Computing metrics...


{'MAP@12': 0.006313156988173997,
 'HitRate@12': 0.0663777107735127,
 'NDCG@12': 0.014016363034196056}

### Collaborative filtering using ALS modelling

The training dataset is massive with around 1.3M unique customers and around 100k unique articles. This makes the collaborative filtering model to run very slowly on the entire dataset as the sparse matrix is massive.

So, down sampling only the last 3 months records would help in running the ALS model faster and also the data would be more relative to recommend as the customer's fashion sense from last 2 years would not matter much.

In [12]:
# Subset the training transactions to only include the last 60 days of data.
cutoff = train_transactions['t_dat'].max() - pd.Timedelta(days=60)
train_transaction_subset = train_transactions[train_transactions['t_dat'] >= cutoff]

In [53]:
print(len(train_transactions.customer_id.unique()), "unique users in the original training set.")
print(len(train_transaction_subset.customer_id.unique()), "unique users in the last 60 days of the training set.")

1356709 unique users in the original training set.
403116 unique users in the last 60 days of the training set.


In [62]:
als_model, user_map, item_map, reverse_item_map, item_user_matrix = train_als_model(
    train_transaction_subset,
    factors=100,
    iterations=15,
    regularization=0.01,
    alpha=40.0,
    random_seed=42
)

als_recommender = create_als_recommender(
    als_model=als_model,
    user_map=user_map,
    item_map=item_map,
    reverse_item_map=reverse_item_map,
    item_user_matrix=item_user_matrix,
    baseline=recency_model
)

1. User map shape: 403116
2. Item map shape: 36899
3. Reverse item map shape: 36899
Training Implicit ALS model...


  0%|          | 0/15 [00:00<?, ?it/s]

Training complete!


In [ ]:
als_file_path = ROOT_DIR / 'saved_models' / 'als_recommender_model.pkl'

In [ ]:
# Save the ALS model and mappings to a file for later use.
model_artifact = {
    'model': als_model,                
    'user_map': user_map,              
    'item_map': item_map,              
    'reverse_item_map': reverse_item_map,
    'item_user_matrix': item_user_matrix, 
}

with open(als_file_path, 'wb') as f:
    pickle.dump(model_artifact, f)

print("ALS model successfully saved!")

ALS model successfully saved!


In [68]:
# Load the ALS model and mappings from the file for evaluation.
with open(als_file_path, "rb") as f:
    artifacts = pickle.load(f)
    
als_recommender = create_als_recommender(
    als_model=artifacts["als_model"],
    user_map=artifacts["user_map"],
    item_map=artifacts["item_map"],
    reverse_item_map=artifacts["reverse_item_map"],
    item_user_matrix=artifacts["item_user_matrix"],
    baseline=recency_model  # Fallback for cold-start or unknown customers
)

KeyError: 'als_model'

In [64]:
evaluate_all(als_recommender, val_transactions)

Generating predictions for 68,984 validation customers...
Preparing evaluation matrices...
Computing metrics...


{'MAP@12': 0.023023678756249187,
 'HitRate@12': 0.1069668328887858,
 'NDCG@12': 0.03580656426402332}